<a href="https://colab.research.google.com/github/ArinShemeem/peft-lora-t5-formality-transfer/blob/main/finetuning.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
import pandas as pd
from sklearn.model_selection import train_test_split
from transformers import T5Tokenizer, T5ForConditionalGeneration,DataCollatorForSeq2Seq
from datasets import Dataset
import torch
from peft import get_peft_model,TaskType,LoraConfig
from torch.optim import AdamW
from tqdm import tqdm
from torch.utils.data import DataLoader

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


In [ ]:
import pandas as pd

file_path = '/content/drive/MyDrive/Colab Notebooks/Formality Transfer Dataset.csv'    #should add the csv file path to work!!!!!
df = pd.read_csv(file_path)

In [ ]:
print(f"Total rows before cleaing: {len(df)}")
df = df.dropna()
df = df.drop_duplicates()
print(f"Total rows after cleaing: {len(df)}")

Total rows before cleaing: 104562
Total rows after cleaing: 104457


In [ ]:
X = df['informal'].tolist()
y = df['formal'].tolist()

In [ ]:
from sklearn.model_selection import train_test_split
X_train,X_test,y_train,y_test=train_test_split(X,y,test_size=0.2)

In [ ]:
"""
from transformers import BartTokenizer, BartForConditionalGeneration
model_name='facebook/bart-large'
tokenizer = BartTokenizer.from_pretrained(model_name)
model = BartForConditionalGeneration.from_pretrained(model_name)
"""

model_name = "t5-small"

tokenizer = T5Tokenizer.from_pretrained(model_name)
model = T5ForConditionalGeneration.from_pretrained(model_name)


/usr/local/lib/python3.11/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(
You are using the default legacy behaviour of the <class 'transformers.models.t5.tokenization_t5.T5Tokenizer'>. This is expected, and simply means that the `legacy` (previous) behavior will be used so nothing changes for you. If you want to use the new behaviour, set `legacy=False`. This should only be set if you understand what it means, and thoroughly read the reason why this was added as explained in https://github.com/huggingface/transformers/pull/24565


In [ ]:
X_train_tokenized = tokenizer(X_train, padding=True, truncation=True, return_tensors="pt",max_length=128)
y_train_tokenized = tokenizer(y_train, padding=True, truncation=True, return_tensors="pt",max_length=128)

In [ ]:
X_test_tokenized = tokenizer(X_test, padding=True, truncation=True, return_tensors="pt",max_length=128)
y_test_tokenized = tokenizer(y_test, padding=True, truncation=True, return_tensors="pt",max_length=128)

In [ ]:
X_train_tokenized['labels'] = y_train_tokenized['input_ids']
X_test_tokenized['labels'] = y_test_tokenized['input_ids']

#tokenized output has input ids and attention mask tensors
#to link the output with ip we use the labels tensor


In [ ]:
train_dataset = Dataset.from_dict({
    'input_ids': X_train_tokenized['input_ids'].tolist(),
    'attention_mask': X_train_tokenized['attention_mask'].tolist(),
    'labels': X_train_tokenized['labels'].tolist()
})

test_dataset = Dataset.from_dict({
    'input_ids': X_test_tokenized['input_ids'].tolist(),
    'attention_mask': X_test_tokenized['attention_mask'].tolist(),
    'labels': X_test_tokenized['labels'].tolist()
})
data_collator = DataCollatorForSeq2Seq(tokenizer=tokenizer, model=model)

train_dataloader = DataLoader(train_dataset, shuffle=True, batch_size=16, collate_fn=data_collator)
val_dataloader = DataLoader(test_dataset, batch_size=16, collate_fn=data_collator)

In [ ]:
'''
informal_text = X_train[0:5]
input_ids = X_train_tokenized["input_ids"][0:5]
attention_mask = X_train_tokenized["attention_mask"][0:5]
label_ids = X_train_tokenized["labels"][0:5]
formal_text = y_train[0:5]
'''

'\ninformal_text = X_train[0:5]\ninput_ids = X_train_tokenized["input_ids"][0:5]\nattention_mask = X_train_tokenized["attention_mask"][0:5]\nlabel_ids = X_train_tokenized["labels"][0:5]\nformal_text = y_train[0:5]\n'

In [ ]:
'''for i in range(1):
  print("=== informal Text ===")
  print(f"Informal (Input): {informal_text[i]}")
  print("\n")
  print("=== Tokenized Input ===")
  print(f"input_ids: {input_ids}")
  print(f"attention_mask: {attention_mask}\n")
  print("\n")
  print("=== Formalized Text ===")
  print(f"Formal (Output): {formal_text[i]}\n")
  print("\n")
  print("=== Tokenized Output ===")
  print(f"labels: {label_ids}")
'''

'for i in range(1):\n  print("=== informal Text ===")\n  print(f"Informal (Input): {informal_text[i]}")\n  print("\n")\n  print("=== Tokenized Input ===")\n  print(f"input_ids: {input_ids}")\n  print(f"attention_mask: {attention_mask}\n")\n  print("\n")\n  print("=== Formalized Text ===")\n  print(f"Formal (Output): {formal_text[i]}\n")\n  print("\n")\n  print("=== Tokenized Output ===")\n  print(f"labels: {label_ids}")\n'

In [ ]:
'''
for i in range(5):
  print(formal_text[i])
  print(label_ids[i])
  print(tokenizer.decode(label_ids[i]))
'''

'\nfor i in range(5):\n  print(formal_text[i])\n  print(label_ids[i])\n  print(tokenizer.decode(label_ids[i]))\n'

In [ ]:
'''
peft_config = LoraConfig(
    task_type=TaskType.SEQ_2_SEQ_LM,
    inference_mode=False,
    r=16,
    lora_alpha=64,
    lora_dropout=0.0
)

model = get_peft_model(model, peft_config)
model.print_trainable_parameters()
'''

trainable params: 589,824 || all params: 61,096,448 || trainable%: 0.9654


In [ ]:

'''
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
model.to(device)

optimizer = AdamW(model.parameters(), lr=1e-4,weight_decay=0.01)

model.train()

for epoch in range(3):
    loop = tqdm(train_dataloader, leave=True)
    for batch in loop:
        batch = {k: v.to(device) for k, v in batch.items()}

        outputs = model(**batch)
        loss = outputs.loss

        loss.backward()
        optimizer.step()
        optimizer.zero_grad()

        loop.set_description(f"Epoch {epoch}")
        loop.set_postfix(loss=loss.item())

    torch.cuda.empty_cache()
  '''

Epoch 2: 100%|██████████| 5223/5223 [12:24<00:00,  7.02it/s, loss=0.433]


In [ ]:
'''
model.save_pretrained("./peft_lora_t5")
tokenizer.save_pretrained("./peft_lora_t5")
'''

('./peft_lora_t5/tokenizer_config.json',
 './peft_lora_t5/special_tokens_map.json',
 './peft_lora_t5/spiece.model',
 './peft_lora_t5/added_tokens.json')

In [ ]:
import evaluate

In [ ]:
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
peft_model_id = "./peft_lora_t5"
model_finetuned = T5ForConditionalGeneration.from_pretrained(peft_model_id).to(device)
tokenizer_finetuned = T5Tokenizer.from_pretrained(peft_model_id)


In [ ]:
base_model_name = 't5-small'
tokenizer_base = T5Tokenizer.from_pretrained(base_model_name)
model_base = T5ForConditionalGeneration.from_pretrained(base_model_name)
data_collator_base = DataCollatorForSeq2Seq(tokenizer=tokenizer_base, model=model_base)
val_dataloader_base = DataLoader(test_dataset, batch_size=16, collate_fn=data_collator_base)

In [ ]:
bleu_metric = evaluate.load("bleu")
rouge_metric = evaluate.load("rouge")
bertscore_metric = evaluate.load("bertscore")

In [ ]:
def evaluate_model(model, tokenizer, dataloader, device, model_name):
    model.eval()
    predictions = []
    references = []
    losses = []

    with torch.no_grad():
        for batch in tqdm(dataloader, desc=f"Evaluating {model_name}"):
            batch = {k: v.to(device) for k, v in batch.items()}

            # Generate predictions
            generated_ids = model.generate(
                input_ids=batch["input_ids"],
                attention_mask=batch["attention_mask"],
                max_length=128,
                num_beams=4,
                early_stopping=True
            )

            # Calculate loss
            outputs = model(**batch)
            loss = outputs.loss
            losses.append(loss.item())

            # Decode predictions and references
            preds = tokenizer.batch_decode(generated_ids, skip_special_tokens=True)
            refs = tokenizer.batch_decode(batch["labels"], skip_special_tokens=True)

            predictions.extend(preds)
            references.extend(refs)

    # Calculate metrics
    bleu_results = bleu_metric.compute(predictions=predictions, references=[[ref] for ref in references])
    rouge_results = rouge_metric.compute(predictions=predictions, references=references)

    # Calculate perplexity
    perplexity = np.exp(np.mean(losses))

    return {
        "model": model_name,
        "bleu": bleu_results["bleu"],
        "rouge1": rouge_results["rouge1"],
        "rouge2": rouge_results["rouge2"],
        "rougeL": rouge_results["rougeL"],
        "perplexity": perplexity,
        "predictions": predictions,
        "references": references
    }

In [ ]:
base_results = evaluate_model(model_base, tokenizer_base, val_dataloader_base, device, "Base T5")

In [ ]:
finetuned_results = evaluate_model(model_finetuned, tokenizer_finetuned, val_dataloader, device, "Fine-Tuned T5")

In [ ]:
print("\nModel Comparison:")
print(f"{'Metric':<15} {'Base T5':<15} {'Fine-Tuned T5':<15}")
print("-" * 45)
for metric in ["bleu", "rouge1", "rouge2", "rougeL", "perplexity"]:
    print(f"{metric:<15} {base_results[metric]:<15.4f} {finetuned_results[metric]:<15.4f}")


Model Comparison:
Metric          Base T5         Fine-Tuned T5  
---------------------------------------------


TypeError: 'NoneType' object is not subscriptable

In [ ]:
# Ensure model is on the right device
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
model_finetuned = model_finetuned.to(device)

# Prepare input and send to same device
sample_input = "And some of them in here, too."
inputs = tokenizer_finetuned(sample_input, return_tensors="pt", padding=True, truncation=True, max_length=128).to(device)

# Generate output
model_finetuned.eval()
with torch.no_grad():
    outputs = model_finetuned.generate(
        input_ids=inputs["input_ids"],
        attention_mask=inputs["attention_mask"],
        max_length=128,
        num_beams=4,
        early_stopping=True
    )

# Decode and print
prediction = tokenizer_finetuned.decode(outputs[0], skip_special_tokens=True)
print("Formalized Output:", prediction)


Formalized Output: And some of them in here, too.


In [ ]:
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
model_base = model_base.to(device)
model_base.eval()
sample_input = "LOL*  Gotta give credit were credit is due!!"
inputs = tokenizer_base(sample_input, return_tensors="pt", padding=True, truncation=True, max_length=128).to(device)

with torch.no_grad():
    outputs = model_base.generate(
        input_ids=inputs["input_ids"],
        attention_mask=inputs["attention_mask"],
        max_length=128,
        num_beams=4,
        early_stopping=True
    )

prediction = tokenizer_base.decode(outputs[0], skip_special_tokens=True)
print("Base T5 Output:", prediction)


Base T5 Output: LOL* Gotta give credit was credit is due!!
